# Automatic Bank Loan Approval — Exploratory Data Analysis & Model Training

This notebook demonstrates dataset inspection, preprocessing, model training, evaluation, and pipeline serialization for the Automatic Bank Loan Approval & Decision Support System.

In [1]:
import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Import custom modules from src
import sys
sys.path.append('../src')
from preprocess import build_preprocessor, NUMERICAL_FEATURES, CATEGORICAL_FEATURES, prepare_input_dataframe
from evaluate import evaluate_model, print_evaluation_summary
from predict import predict_loan

## 1. Load and Inspect Dataset

In [2]:
data_path = '../data/Loan_Data.csv'
df = pd.read_csv(data_path)
print("Dataset Shape:", df.shape)
df.head()

## 2. Preprocessing & Train/Test Split

In [3]:
feature_cols = NUMERICAL_FEATURES + CATEGORICAL_FEATURES
X = df[feature_cols].copy()
X["Credit_History"] = X["Credit_History"].apply(
    lambda x: str(int(x)) if pd.notnull(x) and str(x).strip() != "" else None
)
y = df["Loan_Status"].map({"Y": 1, "N": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

## 3. Candidate Model Evaluation

In [4]:
from sklearn.pipeline import Pipeline

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
}

results = []
for name, clf in models.items():
    pipe = Pipeline([
        ("preprocessor", build_preprocessor()),
        ("classifier", clf)
    ])
    pipe.fit(X_train, y_train)
    metrics = evaluate_model(pipe, X_test, y_test, model_name=name)
    results.append(metrics)

print_evaluation_summary(results)

## 4. Test Single Inference API

In [5]:
sample_input = {
    "dependents": "1",
    "education": "Graduate",
    "self_employed": "No",
    "applicant_income": 5000,
    "coapplicant_income": 1500,
    "loan_amount": 200,
    "loan_amount_term": 360,
    "credit_history": 1,
    "property_area": "Urban"
}
res = predict_loan(sample_input)
print("Sample Backend Payload Prediction:", res)